In [ ]:
from scr.scraper import login_facebook, configure_chrome, read_new_posts_from_all_groups
from scr.setup import config
from scr.parser import select_relevant_apartments
from scr.manual_operations import manually_add_relevant_apartments, review_apartments, generate_messages

# Scraping

## Login to Facebook

1. Open facebook.com
2. If credentials found in `.env` — fill in email and password and click login automatically
3. If no credentials — prompts user to log in manually
4. Waits up to 16 minutes for CAPTCHA to be solved by user (detected by presence of the search bar)

In [ ]:
driver, wait = configure_chrome()
scraper_config = config.system_config.scraper_config

login_facebook(driver=driver, 
               scraper_config=scraper_config,
               wait=wait)

## Retrieve new posts

1. For each monitored Facebook group, determine the cutoff time:
   - Use `last_visited` from config if available, otherwise default to **1 hour ago**
2. Open the group page and scroll down until reaching the cutoff point
3. For each visible new post, extract:
   - `author` — poster's name
   - `created_at` — exact timestamp (via tooltip hover)
   - `text` — post content (expands "See more" if present)
   - `url` — direct link to the post
4. Save all collected posts to `data/raw_data/raw_posts_{current_time}.csv`
5. Update `last_visited` timestamp for each group in `user_config.yaml`

In [ ]:
read_new_posts_from_all_groups(driver=driver)

# Parsing

## Select relevant apartments

1. Load filter criteria from `config.yaml` (price, location, property type, etc.)
2. Load existing `data/relevant_apartments.csv` if it exists
3. Load all raw post CSVs from `data/raw_data/` and delete them after reading
4. Combine all data, drop duplicates by URL, drop rows missing `url` or `text`
5. For each post, send text to LLM (GPT) to extract structured apartment data:
   - price, location, property type, utilities, deposit, move-in date, etc.
   - generates a short English summary
6. Filter extracted apartments against criteria from config
7. Sort by price and publish date
8. Save results to `data/relevant_apartments.csv`

In [ ]:
select_relevant_apartments()

# Manual operations

### Add apartments manually
1. Prompt for location, URL, contact info, price, and comments in a loop
2. Skip duplicate URLs
3. On `Ctrl+C` — stop and save collected entries
4. Merge with existing CSV, deduplicating by URL, keeping the most complete row.
5. Saves it to `data/relevant_apartments.csv`

In [ ]:
manually_add_relevant_apartments()

### Review apartments
1. Load existing apartments from `data/relevant_apartments.csv` one by one
2. For each — show URL and ask whether to keep it
3. Remove rejected entries and save the updated CSV

In [ ]:
review_apartments()

### Generate messages
1. Load message template from `config.yaml`
2. For each apartment in `data/relevant_apartments.csv` — combine template with apartment URL
3. Append contact number + message to `data/messages_{current_time}.txt`, separated by dividers

In [ ]:
generate_messages()